# Controller E v2a development frozen 20261025
Keep private. Same Qwen/NF4, one call/384 tokens. Six fresh fictional behavior checks and two 14k+ token XGrammar/prefill256 checks MUST pass before 800 development E invocations. Full uses cached actual pre-E states. No index download, real gold, selection or final data; no runtime quality scoring, tuning, retries or promotion. Partial outputs always exported.

In [ ]:
from pathlib import Path
import base64,hashlib,io,json,os,subprocess,sys,tarfile,zipfile
matches=[p for p in Path('/kaggle/input').rglob('controller_e_v2a_development_v1.tar.gz.bin') if p.is_file()]
if not matches: matches=[p for p in Path('/kaggle/input').rglob('controller_e_v2a_development_v1.tar.gz') if p.is_file()]
assert len(matches)==1, 'Attach the private frozen E v2a archive dataset'
raw=matches[0].read_bytes()
assert hashlib.sha256(raw).hexdigest()=='54cbca0d57d5d2733afac260c988f47a6bafef77dcef1bd56d3a188c638d1c50'
WORK=Path('/kaggle/working')
BASE=WORK/('e_v2a_'+'54cbca0d57d5'); BASE.mkdir(exist_ok=False)
with tarfile.open(fileobj=io.BytesIO(raw),mode='r:gz') as t:
    for m in t.getmembers():
        assert m.isfile() and (BASE/m.name).resolve().is_relative_to(BASE.resolve())
    t.extractall(BASE,filter='data')
ROOT=BASE/'controller_e_v2a_development_v1'
for rel,h in json.loads((ROOT/'bundle_checksums.json').read_text()).items():
    assert hashlib.sha256((ROOT/rel).read_bytes()).hexdigest()==h,rel
OUTPUT=WORK/'controller_e_v2a_development_v1_outputs'
LOG=WORK/'controller_e_v2a_run.log'
os.environ['OMP_NUM_THREADS']='4'
os.environ['TOKENIZERS_PARALLELISM']='false'
os.chdir(ROOT)
def run(command):
    with LOG.open('a') as log:
        p=subprocess.Popen(command,cwd=ROOT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in p.stdout:
            log.write(line);log.flush();print(line,end='',flush=True)
        if p.wait():raise RuntimeError('Frozen command failed: '+repr(command))
try:
    run([sys.executable,'-m','pip','install','-r','requirements-controller-e-v2a.txt'])
    run([sys.executable,'-m','unittest','tests.test_evidence_v2a','tests.test_controller_v2a_policy','-q'])
    run([sys.executable,'scripts/run_controller_e_v2a.py','--preflight','--output-dir',str(OUTPUT)])
    run([sys.executable,'-u','scripts/run_controller_e_v2a.py','--output-dir',str(OUTPUT)])
finally:
    OUTPUT.mkdir(exist_ok=True)
    if LOG.exists():(OUTPUT/'run.log').write_bytes(LOG.read_bytes())
    (OUTPUT/'input_execution_manifest.json').write_bytes((ROOT/'execution_manifest.json').read_bytes())
    checks={str(p.relative_to(OUTPUT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in OUTPUT.rglob('*') if p.is_file()}
    (OUTPUT/'output_checksums.json').write_text(json.dumps(checks,indent=2))
    archive=WORK/'controller_e_v2a_development_v1_outputs.zip'
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED,strict_timestamps=False) as z:
        for p in sorted(OUTPUT.rglob('*')):
            if p.is_file():z.write(p,str(p.relative_to(OUTPUT)))
    print('Preserved output:',archive,'SHA256:',hashlib.sha256(archive.read_bytes()).hexdigest())
